<a href="https://colab.research.google.com/github/rtovardev/taller-langchain-agentes/blob/main/notebook/taller_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Construye tu primer agente de IA con LangChain

**Taller práctico · EurusConf 2026**

Este notebook es la **Parte 1** del taller y corre en **Google Colab** (no instalas nada en tu máquina).

Vas a construir un **agente investigador** y luego lo vas a hacer **sólido**: que se vea bien (streaming), que recuerde (memoria) y que devuelva datos limpios (salida estructurada).

> En la **Parte 2** pasamos a VS Code y llevamos este mismo agente a un **proyecto real** (con `uv` + `src/`). Eso vive en el repo.

**Cómo correrlo:** ve celda por celda (Shift+Enter). Corre primero el Setup.

## 0. Setup

In [26]:
# Instala las librerías (en Colab). Tarda ~1 min.
%pip install langchain langchain-google-genai langchain-openai langchain-community ddgs python-dotenv pydantic requests tavily

In [27]:
from google.colab import userdata

# Cargando las credenciales desde las variables de entorno
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [28]:
from langchain.chat_models import init_chat_model
from IPython.display import display, Markdown

def consultar_modelo(api_key: str, query: str) -> str:
    """
    Recibe una API Key de OpenRouter y un query, y devuelve la respuesta del modelo.
    """
    model = init_chat_model(
        model="openai/gpt-4o-mini",
        model_provider="openai",
        base_url="https://openrouter.ai/api/v1",
        api_key=api_key
    )

    respuesta = model.invoke(query)

    return respuesta.content

res = consultar_modelo(OPENROUTER_API_KEY, "¿Qué es un agente de IA?")

display(Markdown(res))

Un agente de inteligencia artificial (IA) es un sistema o programa que utiliza técnicas de IA para realizar tareas que requieren inteligencia humana, como el aprendizaje, la toma de decisiones, la percepción y la interacción con el entorno. Estos agentes pueden actuar de manera autónoma o cooperativa y pueden adaptarse a nuevas situaciones y aprender de la experiencia.

Los agentes de IA pueden clasificarse en varias categorías, según su complejidad y funcionalidad:

1. **Agentes Reactivos**: Responden a estímulos del entorno sin mantener un estado interno. Por ejemplo, un chatbot que responde basándose en palabras clave.

2. **Agentes Basados en el Conocimiento**: Utilizan una base de conocimientos para tomar decisiones informadas y resolver problemas.

3. **Agentes de Aprendizaje**: Mejoran su desempeño con el tiempo mediante el aprendizaje a partir de datos y experiencias pasadas, como los sistemas de recomendación.

4. **Agentes Autónomos**: Operan de manera independiente, realizando tareas complejas sin intervención humana continua, como vehículos autónomos.

5. **Agentes Conversacionales**: Incluyen chatbots y asistentes virtuales que interactúan con los usuarios a través de lenguaje natural.

Los agentes de IA tienen aplicaciones en diversos campos, como la atención al cliente, la automatización industrial, la medicina, la educación y más, desempeñando un papel cada vez más importante en la vida cotidiana.

## Recordatorio rápido

Ya vimos la teoría en las diapositivas. Lo esencial para programar:

- Un **agente** = modelo + herramientas + un *loop*: recibe una tarea, decide si usar una herramienta, observa el resultado y repite hasta responder.
- Hoy lo armamos con **`create_agent`** (de LangChain), que por debajo corre sobre **LangGraph**.
- Una **herramienta (`@tool`)** es solo una función de Python con una descripción. El modelo decide cuándo llamarla.

## Actividad 1 — Tu agente investigador

Primero un agente con una herramienta trivial (para ver el *loop*), y luego el de verdad: uno que **busca en la web**.

In [41]:
# Definimos el modelo que vamos a utilizar para la creacion de agentes — Facilitador (Modelo Pago)
model_facilitador = init_chat_model(
    model="openai/gpt-4.1-mini",
    model_provider="openai",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# Definimos el modelo que vamos a utilizar para la creacion de agentes — Estudiante (Modelo Gratuito)
model_estudiante = init_chat_model(
    model="gemini-3.1-flash-lite",
    model_provider="google_genai",
    api_key=GEMINI_API_KEY
)

### ⚠️ DISCLAIMER: Límites de la API Gratuita

Para este taller usaremos la capa gratuita de **Google Gemini**, la cual está limitada estrictamente a:
* **15 peticiones por minuto (RPM)** (El tope crítico).
* **Hasta 500-1500 peticiones por día (RPD)**, dependiendo del modelo [Google AI Studio](https://ai.google.dev/gemini-api/docs/rate-limits).

> **La trampa del Agente:** Un solo "Play" ejecuta entre 2 y 4 peticiones internas. Si spameas el botón tras un error, consumirás tus 15 peticiones en segundos y Google te bloqueará por un minuto *(Error 429)*. Revisa tu código antes de reintentar.

In [30]:
# DEMO — calentamiento: un agente con UNA herramienta simple, para ver el loop
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from datetime import datetime

@tool
def hora_actual(zona: str = "local") -> str:
    """Devuelve la fecha y hora actual."""
    return datetime.now().strftime("%Y-%m-%d %H:%M")

agente = create_agent(
    model=model_facilitador,
    tools=[hora_actual],
    system_prompt="Eres un asistente útil. Usa las herramientas cuando ayuden.",
)

r = agente.invoke({"messages": [{"role": "user", "content": "¿Qué fecha y hora es?"}]})
print(r["messages"][-1].content)

La fecha y hora actual es 26 de junio de 2026, 15:10. ¿En qué más puedo ayudarte?


### Tu turno
Enfoca el investigador a **TU** tema (tu carrera, un hobby) cambiando el `system_prompt`, y hazle una pregunta actual.

In [31]:
# Importando modulos necesarios
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from tavily import TavilyClient


# TODO: crea tu investigador con un system_prompt enfocado a tu tema
print("Buena suerte!")

Buena suerte!


### Solución A1

In [32]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from tavily import TavilyClient
from IPython.display import display, Markdown
from google.colab import userdata
from datetime import datetime

# Inicializamos el objeto de la clase TavilyClient
client = TavilyClient(api_key=userdata.get('TAVILY_API_KEY'))

@tool
def buscar(query: str) -> str:
  """Busca información en tiempo real en internet sobre un tema específico."""
  response = client.search(
    query,
    max_results=3,
  )
  return response['results']

mi_investigador = create_agent(
    model=model_facilitador,
    tools=[buscar],
    system_prompt=f"Eres un investigador experto en tecnología. Responde claro y con datos actuales. Fecha actual: {datetime.now().strftime("%Y-%m-%d %H:%M")}",
)

r = mi_investigador.invoke({"messages": [{"role": "user", "content": "¿Qué novedades hay en inteligencia artificial?"}]})

# Extraemos el contenido asegurándonos de que sea un string para Markdown
contenido_crudo = r["messages"][-1].content

if isinstance(contenido_crudo, list):
    # Si es una lista (formato multi-modal), extraemos solo el texto
    texto_final = "".join([m.get("text", "") for m in contenido_crudo if isinstance(m, dict)])
else:
    texto_final = contenido_crudo

display(Markdown(texto_final))

En 2026, la inteligencia artificial está creciendo rápidamente y presenta varias novedades importantes:

1. La IA se consolida como el motor principal de innovación digital, con herramientas que generan campañas completas de marketing (texto, imagen, video) basadas en datos de comportamiento en tiempo real. Esto impacta mucho en el marketing digital, con personalización y segmentación más avanzada.

2. La IA deja de ser solo una herramienta para convertirse en un verdadero aliado en el trabajo y la creación. En sectores como el desarrollo de software, la IA ahora entiende mejor el contexto y colabora en tareas especializadas bajo supervisión humana.

3. Surgen agentes de IA como asistentes digitales capaces de asumir tareas complejas, lo que impulsa a las organizaciones a robustecer sus sistemas de seguridad para enfrentar nuevas amenazas.

4. Aplicaciones médicas avanzan, por ejemplo, en tratamientos de fertilidad usando IA.

En resumen, en 2026 la IA ya no solo responde preguntas sino que colabora, mejora procesos creativos y productivos, y exige mayor atención a la ética y seguridad en su uso. ¿Quieres que te detalle novedades en un área específica como marketing, salud o desarrollo?

## Actividad 2 — Hazlo sólido

El mismo agente, pero mejor: **streaming** (se ve la respuesta llegar), **memoria** (recuerda la conversación) y **salida estructurada** (devuelve datos limpios, no texto suelto).

### 2.1 Streaming — que se vea la respuesta llegar en vivo

#### Usando 'streaming' para ver como se comporta nuestro agente
En el siguiente ejemplo vamos a aprender a como configurar de manera sencilla que podamos ver *dinamicamente el comportamiento del agente* en tiempo real. Esto nos permite poder tener una **observabilidad** de como se comporta el agente mientras estamos en **desarrollo**.

In [33]:
from langchain.messages import AIMessage, HumanMessage
from langchain.agents import create_agent

from datetime import datetime

agente = create_agent(
    model=model_facilitador,
    tools=[buscar],
    system_prompt=f"Eres un asistente de busqueda experto en inteligencia artificial. Fecha actual: {datetime.now().strftime("%Y-%m-%d %H:%M")}"
)

# En vez de esperar toda la respuesta, la imprimimos token por token
stream = agente.stream_events(
    {"messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]},
    version="v3"
)

for snapshot in stream.values:
  # Cada snapshot contiene todo el estado del agente hasta este punto
  latest_message = snapshot["messages"][-1]
  if latest_message.content:
    if isinstance(latest_message, HumanMessage):
      print(f"User: {latest_message.content}")
    elif isinstance(latest_message, AIMessage):
      print(f"Agent: {latest_message.content}")
  elif latest_message.tool_calls:
    print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")


User: Search for AI news and summarize the findings
Agent: [{'type': 'tool_call', 'id': 'call_hHD5pZKNJw4EmTpHBzu5h6pk', 'name': 'buscar', 'args': {'query': 'latest AI news 2026'}}]
Agent: [{'type': 'text', 'text': 'Aquí tienes un resumen de las últimas noticias sobre inteligencia artificial en 2026:\n\n1. Micron ha impulsado las acciones de chips en el mercado, mientras que algunas empresas están considerando recortes relacionados con la IA. (Fuente: Reuters)\n\n2. Google anunció en mayo de 2026 avances clave en IA con el lanzamiento del modelo Gemini 3.5 y Gemini Omni, enfocados en capacidades avanzadas de razonamiento y creación, marcando el inicio de una era "agentic" en inteligencia artificial. (Fuente: Google Blog)\n\n3. Microsoft destaca siete tendencias importantes en IA para 2026, que incluyen la evolución de la IA como un verdadero socio en el trabajo en equipo, mejoras en la seguridad, un impulso en la investigación y mayor eficiencia en la infraestructura tecnológica. (Fuen

#### Generando una mejor experiencia de usuario usando 'streaming'
Gracias a las funcionalidades de **streaming** de langchain ademas de poder testear nuestro agente y ver como se comporta, tambien podemos generar una ***mejor experiencia UX*** para los usuarios.

In [34]:
from langchain_core.messages import AIMessageChunk

pregunta_streaming = {"messages": [{"role": "user", "content": "Dime en 3 puntos por qué LangChain es útil para crear agentes."}]}

print("Agente respondiendo en tiempo real (OpenRouter):\n")

# Streaming simplificado para modelos que retornan texto limpio
for chunk, meta in mi_investigador.stream(pregunta_streaming, stream_mode="messages"):
    if isinstance(chunk, AIMessageChunk):
        print(chunk.content, end="", flush=True)

print("\n\n[Transmisión finalizada]")

Agente respondiendo en tiempo real (OpenRouter):

LangChain es útil para crear agentes por las siguientes razones:

1. **Integración fluida con modelos de lenguaje**: LangChain facilita la conexión con múltiples modelos de lenguaje, permitiendo que los agentes utilicen capacidades avanzadas de comprensión y generación de texto para interactuar de manera más natural y efectiva.

2. **Manejo de flujos de trabajo complejos**: Permite construir cadenas de acciones y lógicas condicionales, lo que hace posible que los agentes realicen tareas complejas, tomen decisiones en función del contexto y coordinen múltiples pasos de manera autónoma.

3. **Extensibilidad y soporte para componentes externos**: LangChain soporta la incorporación de fuentes de datos externas, APIs y bases de conocimiento, lo que aumenta la funcionalidad de los agentes al poder acceder y procesar información en tiempo real o personalizada.

[Transmisión finalizada]


### 2.2 Memoria — que recuerde la conversación
Con un `checkpointer` y un `thread_id`, el agente recuerda lo que ya hablaron.

In [35]:
from langgraph.checkpoint.memory import InMemorySaver

investigador_memoria = create_agent(
    model=model_facilitador,
    tools=[buscar],
    system_prompt="Eres un investigador que recuerda la conversación.",
    checkpointer=InMemorySaver(),
)
config = {"configurable": {"thread_id": "demo-1"}}
investigador_memoria.invoke({"messages": [{"role": "user", "content": "Me interesan los agentes de IA."}]}, config)
r = investigador_memoria.invoke({"messages": [{"role": "user", "content": "¿Qué tema te dije que me interesa?"}]}, config)
print(r["messages"][-1].content)  # debería recordarlo

Me dijiste que te interesan los agentes de inteligencia artificial (IA).


### 2.3 Salida estructurada — datos limpios en vez de texto suelto
Útil para integrar con otros sistemas (guardar en una base, mandar a una API...).

In [36]:
from pydantic import BaseModel, Field

class Resumen(BaseModel):
    tema: str = Field(description="el tema consultado")
    puntos_clave: list[str] = Field(description="3 a 5 puntos clave, en frases cortas")

estructurado = model_facilitador.with_structured_output(Resumen)
print(estructurado.invoke("Resume en puntos clave qué es LangChain"))

tema='LangChain' puntos_clave=['Es un framework para desarrollo de aplicaciones con modelos de lenguaje (LLMs).', 'Facilita la integración de LLMs con otras fuentes de datos y APIs.', 'Permite construir cadenas o flujos de llamadas a modelos y procesos relacionados.', 'Ofrece herramientas para manejo de memoria y contexto durante las conversaciones.', 'Es muy usado para crear chatbots, agentes inteligentes y sistemas conversacionales avanzados.']


### Tu turno
Dale **memoria** a tu investigador (con tu propio `thread_id`) y compruébalo en dos turnos.

In [37]:
# TODO: crea tu investigador con checkpointer=InMemorySaver(), define un config con tu thread_id,
# invócalo dos veces y comprueba que recuerda algo del primer turno.

### Solución A2 — Sin Streaming

In [42]:
# Funciones & Clases — LangChain Framework
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.utils.uuid import uuid7

# Funciones agregadas — Otros
from tavily import TavilyClient
from IPython.display import display, Markdown
from google.colab import userdata
from datetime import datetime


# Inicializamos el objeto de la clase TavilyClient
client = TavilyClient(api_key=userdata.get('TAVILY_API_KEY'))

@tool
def buscar(query: str) -> str:
  """Busca información en tiempo real en internet sobre un tema específico."""
  response = client.search(
    query,
    max_results=3,
  )
  return response['results']

mi_investigador = create_agent(
    model=model_facilitador,
    tools=[buscar],
    system_prompt=f"Eres un investigador experto en tecnología. Responde claro y con datos actuales. Fecha actual: {datetime.now().strftime("%Y-%m-%d %H:%M")}",
    checkpointer=InMemorySaver()
)

config = {"configurable": {"thread_id": str(uuid7())}}

mi_investigador.invoke({"messages": [{"role": "user", "content": "Me encantan los agentes de IA!"}]}, config) # Historial de la conversacion

# Respuesta que vamos a imprimir
r = mi_investigador.invoke({"messages": [{"role": "user", "content": """Segun el tema que te dije que me encanta, realiza una investigacion
de los mejores frameworks para creacion de agentes hoy en dia, haz un ranking teniendo en cuenta cuales son los mejores para aprender y mas versatiles."""}]}, config)



# Extraemos el contenido asegurándonos de que sea un string para Markdown
contenido_crudo = r["messages"][-1].content

if isinstance(contenido_crudo, list):
    # Si es una lista (formato multi-modal), extraemos solo el texto
    texto_final = "".join([m.get("text", "") for m in contenido_crudo if isinstance(m, dict)])
else:
    texto_final = contenido_crudo

display(Markdown(texto_final))

He revisado información actualizada sobre los mejores frameworks para creación de agentes de inteligencia artificial en 2026, y aquí te presento un ranking basado en su versatilidad, facilidad para aprender y popularidad:

1. **LangChain**  
   - Lenguaje: Python  
   - Ventajas: Es de código abierto, muy modular y permite encadenar múltiples componentes y servicios. Ideal para crear agentes poderosos listos para producción con modelos de lenguaje grandes (LLMs).  
   - Uso: Muy popular para aprendizaje y proyectos reales, con amplia comunidad y documentación.

2. **FlowHunt**  
   - Características: Plataforma sin código con capacidades avanzadas para crear agentes de IA empresariales. Despliegue rápido y buena integración con ecosistemas corporativos.  
   - Ventajas: Ideal para usuarios sin conocimiento profundo en programación que quieren soluciones rápidas y escalables.  
   - Popularidad creciente en entornos profesionales.

3. **CrewAI**  
   - Enfoque: Facilita la orquestación de agentes autónomos con un ciclo completo de "Observar-Orientar-Decidir-Actuar" (OODA).  
   - Ventajas: Comunidad activa y enfoque en experiencia del desarrollador, lo que lo hace recomendado para quienes desean comenzar a crear agentes complejos.

4. **Marcos con soporte para RAG (Generación Aumentada por Recuperación)**  
   - Estos frameworks permiten que el agente acceda y utilice información actualizada y precisa, fundamental para aplicaciones empresariales.  
   - Son clave para agentes que necesitan fundamentos en datos en tiempo real.

Resumen para aprender y versatilidad:  
- Para programación y proyectos con flexibilidad: **LangChain** es la opción top.  
- Para usuarios sin código y empresas: **FlowHunt** destaca.  
- Para desarrolladores que buscan experiencia con agentes autónomos avanzados: **CrewAI** es muy recomendable.

¿Quieres que te prepare ejemplos o recursos para aprender con alguno de estos frameworks?

## ¿Por qué construir tu propio agente si Claude Code ya existe?

Para tareas generales, usa lo que ya existe. Construyes el **tuyo** cuando tienes un flujo específico y repetible que se beneficia de:

- **Control:** tú decides qué herramientas tiene y qué puede hacer.
- **Integración:** se conecta a **tus** sistemas (tu base de datos, tu API, tu Notion).
- **Costo y escala:** corre desatendido con el modelo más barato que sirva.
- **Automatización real:** se dispara por eventos, no es un chat.

## Parte 2 — del notebook al proyecto real (en VS Code)

Hasta aquí, todo en el notebook. Ahora llevamos **este mismo agente** a un **proyecto real** con la estructura que usa LangChain:

```
src/agente/
  agent.py      # create_agent(model, tools, system_prompt, checkpointer)
  tools.py      # las @tool
  prompts.py    # el system prompt
main.py         # punto de entrada (agent.invoke)
.env  ·  pyproject.toml  ·  uv.lock
```

Lo hacemos juntos en el repo con **`uv`** (el gestor de paquetes moderno). El paso a paso está en el **README**.

### Si te sobra tiempo: 3 ideas para tu propio agente
Construye uno tuyo en ese proyecto, conectado a algo real (todas con token, sin OAuth):

1. **Investigador + scraping** — busca y además extrae info de una página.
2. **Tu productividad** — Notion o Todoist: convierte lo que escribes en tareas/notas.
3. **Datos en vivo** — una API pública (clima, finanzas, noticias) + memoria.

El código de ejemplo de cada idea está en el **README** del repo.

---

**Recursos:** https://docs.langchain.com · **MART Automations:** https://martautomations.com